# SSAC 2026 Hackathon Analysis
## Power Players Research: Female Football Fan Insights

**Challenge:** How can live fan engagement during FIFA World Cup 2026 become a gateway to lifelong fandom for the female football fan?

**Data:** Wasserman x Sloan MIT cross-tabulated survey (~8,700 women respondents)

In [ ]:
import sys, os
sys.path.insert(0, '../scripts')

from parse_survey import load_survey, get_question, list_questions, get_pct, get_counts, aggregate_rankings
from survey_analysis import *
from visualization import (
    PALETTE, BACKGROUND, TEXT_COLOR, GRID_COLOR,
    _apply_dark_theme, _save_if_needed,
    plot_segment_bars, plot_gap_chart, plot_index_heatmap,
    plot_single_bar, plot_stacked_comparison,
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# Apply dark theme globally
_apply_dark_theme()
BG = BACKGROUND
TEXT = TEXT_COLOR
GRID = GRID_COLOR

pd.set_option('display.max_colwidth', 60)
os.makedirs('../outputs', exist_ok=True)

print('Setup complete')

In [ ]:
# Load and parse the survey
xlsx = '../problem and data/Power Players Research - Wasserman x Sloan MIT research.xlsx'
survey = load_survey(xlsx)

# Quick inventory
q_list = list_questions(survey)
print(f'Parsed {len(q_list)} questions from {survey["raw_df"].shape[0]} rows')
print(f'\nTotals per dimension:')
for dim, counts in survey['totals'].items():
    print(f'  {dim}: {counts}')

## 1. The Commercial Power: Decision-Maker Status

In [ ]:
# Q12: Purchase decision-maker role
pct12 = compare_segments(survey, 'Q12', 'motherhood')
print('Q12: Household Purchase Decision Role')
print(pct12.to_string())

dm = decision_maker_pct(survey)
print(f"\n>>> {dm['combined_pct']}% of women are primary or shared purchase decision-makers <<<")
print(f"    Primary: {dm['primary_pct']}% | Shared: {dm['shared_pct']}%")
print(f"    Moms primary: {compare_segments(survey, 'Q12', 'motherhood').iloc[0]['Mom']:.1f}%")
print(f"    Non-Moms primary: {compare_segments(survey, 'Q12', 'motherhood').iloc[0]['Non-Mom']:.1f}%")

In [ ]:
# Q30: Sports-related purchase decision role
try:
    pct30 = compare_segments(survey, 'Q30', 'sports_fan')
    print('Q30: Sports Purchase Decision Role (Sports Fans Only)')
    print(pct30.to_string())
except Exception as e:
    print(f'Q30 error: {e}')

# Visualization: Q12 decision-maker comparison (Mom vs Non-Mom vs Total)
pct12_sorted = pct12.sort_values('Total', ascending=False)
fig = plot_segment_bars(
    pct12_sorted,
    title='94% of Women Are Household Purchase Decision-Makers',
    subtitle='Q12: Who makes purchasing decisions in your household?',
    top_n=5,
    save_path='../outputs/q12_decision_maker.png',
)
plt.show()

## 2. Sports Engagement: Frequency & Leagues

In [ ]:
# Q28: Sports consumption frequency
pct28 = compare_segments(survey, 'Q28', 'motherhood')
print('Q28: How often do you watch/attend/follow sports?')
print(pct28.to_string())

# Key stat: What % watch at least occasionally?
for col in ['Total', 'Mom', 'Non-Mom']:
    engaged = pct28.loc[~pct28.index.str.contains('Never|Rarely', case=False), col].sum()
    print(f'  {col} watching occasionally+: {engaged:.1f}%')

In [ ]:
# Visualization: Mom vs Non-Mom sports frequency
fig = plot_segment_bars(
    pct28[['Mom', 'Non-Mom']],
    title='Moms Are MORE Engaged Sports Fans Than Non-Moms',
    subtitle='Q28: How often do you watch/attend/follow sports?',
    highlight_col='Mom',
    save_path='../outputs/q28_sports_frequency.png',
)
plt.show()

# Gap chart: Mom vs Non-Mom on sports frequency
gap28 = gap_analysis(survey, 'Q28', 'motherhood', 'Mom', 'Non-Mom')
fig = plot_gap_chart(
    gap28, 'Mom', 'Non-Mom',
    title='Mom vs Non-Mom: Sports Engagement Gap',
    save_path='../outputs/q28_mom_gap.png',
)
plt.show()

In [ ]:
# Q29: Which sports leagues — by age cohort
try:
    pct29 = compare_segments(survey, 'Q29', 'age')
    age_cols = [c for c in pct29.columns if pct29[c].sum() > 0]
    top_leagues = pct29[age_cols].sort_values(age_cols[0], ascending=False).head(12)
    print('Q29: Sports Leagues Watched (by Age)')
    print(top_leagues.to_string())
    
    # WNBA Gen Z signal
    if 'WNBA' in pct29.index and '18-28' in pct29.columns:
        wnba_young = pct29.loc['WNBA', '18-28']
        wnba_total = pct29.loc['WNBA', 'Total']
        print(f'\n>>> WNBA: 18-28 = {wnba_young:.1f}% vs Total = {wnba_total:.1f}% (Gen Z over-indexes by {wnba_young/wnba_total*100 - 100:.0f}%) <<<')
except Exception as e:
    print(f'Q29 error: {e}')

In [ ]:
# Visualization: Top leagues by age — grouped bar
try:
    fig, ax = plt.subplots(figsize=(14, 7))
    leagues = top_leagues.head(8).index
    age_groups = ['18-28', '29-44', '45-60']
    x = np.arange(len(leagues))
    w = 0.25
    
    for i, ag in enumerate(age_groups):
        if ag in top_leagues.columns:
            vals = [top_leagues.loc[l, ag] if l in top_leagues.index else 0 for l in leagues]
            bars = ax.bar(x + i*w, vals, w, label=ag, color=PALETTE[i], alpha=0.85)
            for bar, val in zip(bars, vals):
                if val > 2:
                    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                            f'{val:.0f}%', ha='center', fontsize=8, color=TEXT)
    
    ax.set_xticks(x + w)
    ax.set_xticklabels(leagues, fontsize=9, rotation=30, ha='right')
    ax.set_ylabel('% of Sports Fans')
    ax.set_title('Sports League Viewership by Age: WNBA Surges Among 18-28', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig('../outputs/q29_leagues_by_age.png', dpi=300, bbox_inches='tight', facecolor=BG)
    plt.show()
except Exception as e:
    print(f'Chart error: {e}')

## 3. What Women Want from Sports Orgs

In [ ]:
# Q37: How sports orgs could better engage women
try:
    pct37 = compare_segments(survey, 'Q37', 'motherhood')
    sorted37 = pct37.sort_values('Total', ascending=False)
    print('Q37: How could sports organizations better engage women fans?')
    print(sorted37.to_string())
except Exception as e:
    print(f'Q37 error: {e}')

In [ ]:
# Visualization: Q37 — what women want from sports orgs
try:
    fig = plot_single_bar(
        sorted37['Total'],
        title='What Women Want: Better Coverage, Dedicated Spaces, Relevant Partnerships',
        top_n=8,
        save_path='../outputs/q37_engagement_improvements.png',
    )
    plt.show()
except Exception as e:
    print(f'Chart error: {e}')

# Q38: Event motivations bar chart
try:
    pct38 = compare_segments(survey, 'Q38', 'motherhood')
    fig = plot_single_bar(
        pct38.sort_values('Total', ascending=False)['Total'],
        title='Why Women Attend: Entertainment, Team Support, Social Fun',
        top_n=8,
        save_path='../outputs/q38_event_motivations.png',
    )
    plt.show()
except Exception as e:
    print(f'Q38 chart error: {e}')

# Q39: Ranked experiences bar chart
try:
    pct39 = compare_segments(survey, 'Q39', 'motherhood')
    fig = plot_single_bar(
        pct39.sort_values('Total', ascending=False)['Total'],
        title='Most Valued Experiences: Giveaways, Meet & Greets, Entertainment',
        top_n=8,
        save_path='../outputs/q39_experiences.png',
    )
    plt.show()
except Exception as e:
    print(f'Q39 chart error: {e}')

In [ ]:
# Q38: Event attendance motivations
try:
    pct38 = compare_segments(survey, 'Q38', 'motherhood')
    print('Q38: What motivates you most to attend sporting events?')
    print(pct38.sort_values('Total', ascending=False).to_string())
except Exception as e:
    print(f'Q38 error: {e}')

# Q39: Rank sports experiences
try:
    pct39 = compare_segments(survey, 'Q39', 'motherhood')
    print('\nQ39: Rank these sports experiences by importance:')
    print(pct39.sort_values('Total', ascending=False).to_string())
except Exception as e:
    print(f'Q39 error: {e}')

## 4. Brand Perception & Discovery

In [ ]:
# Q16: Brand authenticity importance
try:
    pct16 = compare_segments(survey, 'Q16', 'motherhood')
    print('Q16: How important is brand authenticity?')
    print(pct16.to_string())
    
    # Stacked comparison: brand authenticity across segments
    fig = plot_stacked_comparison(
        pct16,
        title='Brand Authenticity Is Non-Negotiable for Women',
        save_path='../outputs/q16_brand_authenticity.png',
    )
    plt.show()
except Exception as e:
    print(f'Q16 error: {e}')

# Q17: How well brands understand women
try:
    pct17 = compare_segments(survey, 'Q17', 'motherhood')
    print('\nQ17: How well do brands understand you personally?')
    print(pct17.to_string())
except Exception as e:
    print(f'Q17 error: {e}')

# Q18: Ways brands don't understand women
try:
    pct18 = compare_segments(survey, 'Q18', 'motherhood')
    sorted18 = pct18.sort_values('Total', ascending=False)
    print('\nQ18: In what ways do brands NOT understand women?')
    print(sorted18.to_string())
    
    fig = plot_single_bar(
        sorted18['Total'],
        title='How Brands Fail Women: Products, Stereotypes, Marketing',
        top_n=8,
        save_path='../outputs/q18_brand_gaps.png',
    )
    plt.show()
except Exception as e:
    print(f'Q18 error: {e}')

In [ ]:
# Q31: How well sports organizations understand women
try:
    pct31 = compare_segments(survey, 'Q31', 'motherhood')
    print('Q31: How well do sport organizations understand you?')
    print(pct31.to_string())
    
    # Side-by-side: Q17 (brands) vs Q31 (sports orgs) — "don't understand"
    fig = plot_stacked_comparison(
        pct31,
        title='Sports Orgs Also Failing: 33% Say "Don\'t Understand Me"',
        save_path='../outputs/q31_sports_org_perception.png',
    )
    plt.show()
except Exception as e:
    print(f'Q31 error: {e}')

# Q33-Q36: Men's vs Women's sports leagues performance
print('\n' + '=' * 60)
print('GENDER GAP IN SPORTS COVERAGE')
print('=' * 60)
for qid in ['Q33', 'Q34', 'Q35', 'Q36']:
    try:
        pct_q = compare_segments(survey, qid, 'motherhood')
        q = get_question(survey, qid)
        print(f'\n{q["label"][:80]}')
        print(pct_q.to_string())
    except Exception as e:
        print(f'{qid} error: {e}')

## 5. Values & Identity

In [ ]:
# Q8: Personal values rankings
# These are spread across Q8_0_1_RANK through Q8_0_18_RANK
# Let's look at what options are available
q8_ids = [q for q in survey['question_order'] if q.startswith('Q8_')]
print(f'Q8 sub-questions: {len(q8_ids)}')
for qid in q8_ids[:5]:
    q = get_question(survey, qid)
    print(f'  {qid}: options={q["options"][:3]}...')

# Try aggregate rankings
print('\nQ8 Aggregated Rankings (weighted score):')  
q8_ranks = aggregate_rankings(survey, 'Q8')
if 'motherhood' in q8_ranks:
    print(q8_ranks['motherhood'].to_string())

In [ ]:
# Q9: Recognition preference
try:
    pct9 = compare_segments(survey, 'Q9', 'motherhood')
    print('Q9: Would you rather be recognized for?')
    print(pct9.to_string())
except Exception as e:
    print(f'Q9 error: {e}')

# Q10: Free time preference
try:
    pct10 = compare_segments(survey, 'Q10', 'motherhood')
    print('\nQ10: Would you rather spend your free time?')
    print(pct10.to_string())
except Exception as e:
    print(f'Q10 error: {e}')

In [ ]:
# Q19: Brand support of social issues importance
try:
    pct19 = compare_segments(survey, 'Q19', 'motherhood')
    print('Q19: How important is it that brands support social issues?')
    print(pct19.to_string())
except Exception as e:
    print(f'Q19 error: {e}')

# Q20: Most authentic way for brands to support social issues
try:
    pct20 = compare_segments(survey, 'Q20', 'motherhood')
    print('\nQ20: Most authentic way for brands to support social issues?')
    print(pct20.sort_values('Total', ascending=False).to_string())
except Exception as e:
    print(f'Q20 error: {e}')

## 6. Brand Discovery & Content Preferences

In [ ]:
# Q24: How women discover brands (ranking sub-questions)
q24_ids = [q for q in survey['question_order'] if q.startswith('Q24_')]
print(f'Q24 sub-questions: {len(q24_ids)}')
for qid in q24_ids:
    q = get_question(survey, qid)
    print(f'  {qid}: {q["label"][:60]}... | options: {q["options"]}')

# Aggregate rankings
print('\nQ24 Brand Discovery Rankings:')
q24_ranks = aggregate_rankings(survey, 'Q24')
if 'motherhood' in q24_ranks:
    print(q24_ranks['motherhood'].to_string())

In [ ]:
# Q25: Promotional content preferences (ranking sub-questions)
q25_ids = [q for q in survey['question_order'] if q.startswith('Q25_')]
print(f'Q25 sub-questions: {len(q25_ids)}')

print('\nQ25 Promotional Content Rankings:')
q25_ranks = aggregate_rankings(survey, 'Q25')
if 'motherhood' in q25_ranks:
    print(q25_ranks['motherhood'].to_string())

In [ ]:
# Q26: In-person brand engagement preference
try:
    pct26 = compare_segments(survey, 'Q26', 'motherhood')
    print('Q26: How do you prefer to engage with brands in person?')
    print(pct26.sort_values('Total', ascending=False).to_string())
except Exception as e:
    print(f'Q26 error: {e}')

## 7. Merchandise & Commercial Opportunity

In [ ]:
# Q40: Merchandise interest
try:
    pct40 = compare_segments(survey, 'Q40', 'motherhood')
    sorted40 = pct40.sort_values('Total', ascending=False)
    print('Q40: What type of sports merchandise are you most interested in?')
    print(sorted40.to_string())
    
    interested = 100 - sorted40.loc[sorted40.index.str.contains('not interested', case=False), 'Total'].sum()
    print(f'\n>>> {interested:.1f}% of female sports fans ARE interested in merchandise <<<')
except Exception as e:
    print(f'Q40 error: {e}')

In [ ]:
# Visualization: Merchandise interest
try:
    data40 = sorted40[~sorted40.index.str.contains('Other|N/A', case=False)]
    not_interested_opts = [o for o in data40.index if 'not interested' in o.lower()]
    fig = plot_single_bar(
        data40['Total'],
        title='72% of Female Sports Fans Want Merchandise',
        top_n=8,
        highlight_idx=not_interested_opts,
        save_path='../outputs/q40_merchandise.png',
    )
    plt.show()
except Exception as e:
    print(f'Chart error: {e}')

## 8. Mom vs Non-Mom Deep Dive

In [ ]:
# Key gaps between Moms and Non-Moms
questions_to_compare = ['Q12', 'Q13', 'Q16', 'Q17', 'Q19', 'Q28']

print('MOM vs NON-MOM GAP ANALYSIS')
print('=' * 70)
for qid in questions_to_compare:
    try:
        gap = gap_analysis(survey, qid, 'motherhood', 'Mom', 'Non-Mom')
        q = get_question(survey, qid)
        label = q['label'][:60]
        biggest = gap.iloc[0]
        print(f'\n{qid}: {label}')
        print(f'  Biggest gap: "{gap.index[0][:50]}" -> Mom={biggest["Mom"]:.1f}% vs Non-Mom={biggest["Non-Mom"]:.1f}% (gap={biggest["gap_pp"]:+.1f}pp)')
    except Exception as e:
        print(f'{qid}: error - {e}')

## 9. Age Cohort Analysis (18-28 = FIFA WC 2026 Target)

In [ ]:
# Age comparison for key questions
age_questions = ['Q12', 'Q17', 'Q28', 'Q29']

print('AGE COHORT ANALYSIS')
print('=' * 70)
for qid in age_questions:
    try:
        pct = compare_segments(survey, qid, 'age')
        active_cols = [c for c in pct.columns if pct[c].sum() > 0]
        q = get_question(survey, qid)
        print(f'\n{q["label"][:60]}')
        print(pct[active_cols].to_string())
    except Exception as e:
        print(f'{qid}: error - {e}')

# Index heatmap: Mom over-indexing across key questions
try:
    fig = plot_index_heatmap(
        survey,
        ['Q12', 'Q16', 'Q17', 'Q19', 'Q28'],
        dimension='motherhood',
        title='Mom Index: Over/Under-Indexing vs Total Population',
        save_path='../outputs/index_heatmap_motherhood.png',
    )
    if fig:
        plt.show()
except Exception as e:
    print(f'Index heatmap error: {e}')

## 10. SYNTHESIS: Key Insights for Presentation

In [ ]:
print('=' * 70)
print('TOP INSIGHTS FOR PRESENTATION')
print('=' * 70)

insights = generate_quick_insights(survey)
for i, ins in enumerate(insights, 1):
    print(f'\n{i}. [{ins["source"]}] {ins["insight"]}')

print('\n' + '=' * 70)
print('STORY ARC')
print('=' * 70)
print('''
1. THE POWER: 94% of women are purchase decision-makers. This is not a niche
   audience. This is the majority of household spending.

2. THE ENGAGEMENT: 52% of women follow sports at least occasionally.
   Moms are MORE engaged than non-moms (31% often/very often vs 18%).

3. THE GAP: Sports organizations are failing women fans. #1 ask: "Better
   coverage of women's sports." They want representation, not pink merch.

4. THE OPPORTUNITY: 72% of female sports fans want merchandise. WNBA is
   surging among 18-28 year olds. Gen Z is the growth lever.

5. THE SOLUTION: A fan engagement concept that listens, personalizes, and
   converts World Cup attention into lifelong fandom.
''')

In [ ]:
# Export all key DataFrames for presentation reference
import os
os.makedirs('../outputs', exist_ok=True)

# Save key comparison tables
try:
    compare_segments(survey, 'Q12', 'motherhood').to_csv('../outputs/q12_decision_maker.csv')
    compare_segments(survey, 'Q28', 'motherhood').to_csv('../outputs/q28_sports_frequency.csv')
    compare_segments(survey, 'Q37', 'motherhood').to_csv('../outputs/q37_engagement.csv')
    compare_segments(survey, 'Q40', 'motherhood').to_csv('../outputs/q40_merchandise.csv')
    print('Key tables exported to outputs/')
except Exception as e:
    print(f'Export error: {e}')

print('\nAnalysis complete. Time to build the presentation.')